In [34]:
import numpy as np

def compute_topology_overlaps(G2, G4):
    """
    Computes the topological invariants I_22, I_24, I_44 based on set overlaps.
    I_alpha_beta counts how many sets share IDENTICAL elements.
    """
    # Helper to count identical sets
    def count_matches(list_a, list_b):
        matches = 0
        # Convert to sorted tuples to ensure order doesn't affect equality
        set_b = set(tuple(sorted(x)) for x in list_b)
        for item in list_a:
            if tuple(sorted(item)) in set_b:
                matches += 1
        return matches

    # For standard LABS/Ising chains, these overlaps are often 0 or specific integers
    # We implement the general counting logic here.
    I_22 = count_matches(G2, G2) # Self overlap is just len(G2)
    I_44 = count_matches(G4, G4) # Self overlap is just len(G4)
    I_24 = 0 # 2-body set vs 4-body set overlap usually 0 as sizes differ
    
    return {'22': I_22, '44': I_44, '24': I_24}

from math import sin, cos, pi

def compute_theta(dt, total_time, N, G2, G4, lam, lam_dot):
    """
    Computes theta(t) using the analytical solutions for Gamma1 and Gamma2.
    """
    
    # ---  Better Schedule (Trigonometric) ---
    # lambda(t) = sin^2(pi * t / 2T)
    # lambda_dot(t) = (pi / 2T) * sin(pi * t / T)
    
    if total_time == 0:
        return 0.0
    
    
    # ---  Calculate Gamma Terms (LABS assumptions: h^x=1, h^b=0) ---
    # For G2 (size 2): S_x = 2
    # For G4 (size 4): S_x = 4
    
    # Gamma 1 (Eq 16)
    # Gamma1 = 16 * Sum_G2(S_x) + 64 * Sum_G4(S_x)
    term_g1_2 = 16 * len(G2) * 2
    term_g1_4 = 64 * len(G4) * 4
    Gamma1 = term_g1_2 + term_g1_4
    
    # Gamma 2 (Eq 17)
    # G2 term: Sum (lambda^2 * S_x)
    # S_x = 2
    sum_G2 = len(G2) * (lam**2 * 2)
    
    # G4 term: 4 * Sum (4*lambda^2 * S_x + (1-lambda)^2 * 8)
    # S_x = 4
    # Inner = 16*lam^2 + 8*(1-lam)^2
    sum_G4 = 4 * len(G4) * (16 * (lam**2) + 8 * ((1 - lam)**2))
    
    # Topology part
    I_vals = compute_topology_overlaps(G2, G4)
    term_topology = 4 * (lam**2) * (4 * I_vals['24'] + I_vals['22']) + 64 * (lam**2) * I_vals['44']
    
    # Combine Gamma 2
    Gamma2 = -256 * (term_topology + sum_G2 + sum_G4)

    # ---  Alpha & Theta ---
    if abs(Gamma2) < 1e-12:
        alpha = 0.0
    else:
        alpha = - Gamma1 / Gamma2
        
    return dt * alpha * lam_dot

def test():
    @cudaq.kernel
    def two_qubit_layer(theta, q1, q2):
        rx(np.pi/2, q1)
        
        cx(q1, q2)
        rz(theta, q2)
        cx(q1, q2)
        
        rx(np.pi/2, qubits[q1])
        rx(-np.pi/2, q2)
        
        cx(q1, q2)
        rz(theta, q2)
        cx(q1, q2)
        
        rx(-np.pi/2, q1)

In [42]:
import cudaq
import time

######################### JENGA ##################################

@cudaq.kernel
def qc(n: int, dt: float, indices: list[int], indices2: list[int], theta_list_CD: list[float], lamb_list_AD: list[float], CD: list[bool], AD: list[bool]):
    qubits = cudaq.qvector(n)
    
    # Apply gates
    for i in range(n):
        h(qubits[i])

    for t in range(len(theta_list_CD)):
        if CD[t] == True:
            for i in range(len(indices) // 2):
                i1 = indices[2*i]
                i0 = indices[2*i + 1]
        
                rx(np.pi/2, qubits[i0])
        
                cx(qubits[i0], qubits[i1])
                rz(theta_list_CD[t], qubits[i1])
                cx(qubits[i0], qubits[i1])
        
                rx(-np.pi/2, qubits[i0])
                rx(np.pi/2, qubits[i1])
        
                cx(qubits[i0], qubits[i1])
                rz(theta_list_CD[t], qubits[i1])
                cx(qubits[i0], qubits[i1])
        
                rx(-np.pi/2, qubits[i1])
        
            for i in range(len(indices2) // 4):
                i0 = indices2[4*i]
                i1 = indices2[4*i + 1]
                i2 = indices2[4*i + 2]
                i3 = indices2[4*i + 3]
        
                rx(-np.pi/2, qubits[i0])
                ry(np.pi/2, qubits[i1])
                ry(-np.pi/2, qubits[i2])
        
                cx(qubits[i0], qubits[i1])
                rz(-np.pi/2, qubits[i1])
                cx(qubits[i0], qubits[i1])
        
                cx(qubits[i2], qubits[i3])
                rz(-np.pi/2, qubits[i3])
                cx(qubits[i2], qubits[i3])
        
                rx(np.pi/2, qubits[i0])
                ry(-np.pi/2, qubits[i1])
                ry(np.pi/2, qubits[i2])
                rx(-np.pi/2, qubits[i3])
        
                rx(-np.pi/2, qubits[i1])
                rx(-np.pi/2, qubits[i2])
        
                cx(qubits[i1], qubits[i2])
                rz(theta_list_CD[t], qubits[i2])
                cx(qubits[i1], qubits[i2])
        
                rx(np.pi/2, qubits[i1])
                rx(np.pi, qubits[i2])
        
                ry(np.pi/2, qubits[i1])
        
                cx(qubits[i0], qubits[i1])
                rz(np.pi/2, qubits[i1])
                cx(qubits[i0], qubits[i1])
        
                rx(np.pi/2, qubits[i0])
                ry(-np.pi/2, qubits[i1])
        
                cx(qubits[i1], qubits[i2])
                rz(-theta_list_CD[t], qubits[i2])
                cx(qubits[i1], qubits[i2])
        
                rx(np.pi/2, qubits[i1])
                rx(-np.pi, qubits[i2])
        
                cx(qubits[i1], qubits[i2])
                rz(-theta_list_CD[t], qubits[i2])
                cx(qubits[i1], qubits[i2])
                
                rx(-np.pi, qubits[i1])
                ry(np.pi/2, qubits[i2])
        
                cx(qubits[i2], qubits[i3])
                rz(-np.pi/2, qubits[i3])
                cx(qubits[i2], qubits[i3])
        
                ry(-np.pi/2, qubits[i2])
                rx(-np.pi/2, qubits[i3])
        
                rx(-np.pi/2, qubits[i2])
        
                cx(qubits[i1], qubits[i2])
                rz(theta_list_CD[t], qubits[i2])
                cx(qubits[i1], qubits[i2])
        
                rx(np.pi/2, qubits[i1])
                rx(np.pi/2, qubits[i2])
                
                ry(-np.pi/2, qubits[i1])
                ry(np.pi/2, qubits[i2])
        
                cx(qubits[i0], qubits[i1])
                rz(np.pi/2, qubits[i1])
                cx(qubits[i0], qubits[i1])
        
                cx(qubits[i2], qubits[i3])
                rz(np.pi/2, qubits[i3])
                cx(qubits[i2], qubits[i3])
        
                ry(np.pi/2, qubits[i1])
                ry(-np.pi/2, qubits[i2])
                rx(np.pi/2, qubits[i3])
    
        if AD[t] == True:
            for i in range(n):
                rx(2*dt - 2*lamb_list_AD[t]*dt, qubits[i])

            
            for i in range(len(indices) // 2):
                i1 = indices[2*i]
                i0 = indices[2*i + 1]

                cx(qubits[i0], qubits[i1])
                rz(4*lamb_list_AD[t]*dt, qubits[i1])
                cx(qubits[i0], qubits[i1])
            
            for i in range(len(indices2) // 4):
                i0 = indices2[4*i]
                i1 = indices2[4*i + 1]
                i2 = indices2[4*i + 2]
                i3 = indices2[4*i + 3]

                cx(qubits[i0],qubits[i1])
                cx(qubits[i1],qubits[i2])
                cx(qubits[i2],qubits[i3])
                rz(8*lamb_list_AD[t]*dt, qubits[i3])
                cx(qubits[i2],qubits[i3])
                cx(qubits[i1],qubits[i2])
                cx(qubits[i0],qubits[i1])
                

    mz(qubits)


n = 7


pairs = []
for i in range(1,n):
    square = []
    for j in range(n-i):
        square += [(i+j,j)]
    
    pairs += [square]

full = []
for x in range(len(pairs)):
    test = pairs[x]
    for i in range(len(test)):
        for j in range(len(test)):
            list1 = [test[i][0], test[i][1], test[j][0], test[j][1]]
            set1 = set()
            for w in list1:
                if list1.count(w) == 1:
                    set1.add(w)
            if set1 != set():
                full += [set1]

unique = [set(s) for s in set(frozenset(s) for s in full)]

list_pairs = []
for i in unique:
    list_pairs += [list(i)]

list_2 = []
list_4 = []
G2 = []
G4 = []

for i in list_pairs:
    if len(i) == 2:
        list_2 += [i[0], i[1]]
        G2 += [i]
    else:
        list_4 += [i[0], i[1], i[2], i[3]]
        G4 += [i]





T=1               # total time
n_steps = 10       # number of trotter steps
n_start_ad = 7
n_end_cd = 8
dt = T / n_steps
N = n

thetas =[]
lamb_list = []

for step in range(1, n_steps + 1):
    t = step * dt

    ########## CUSTOM LAM ############
    arg = (pi * t) / (2.0 * T)
        
    lam = sin(arg)**2
    
    lam_dot = (pi / (2.0 * T)) * sin((pi * t) / T)
    ##################################
    
    theta_val = compute_theta(dt, T, N, G2, G4, lam, lam_dot)
    thetas.append(theta_val)
    lamb_list.append(lam)

CD_bool = []
for i in range(n_end_cd):
    CD_bool += [True]

for i in range(n_end_cd, n_steps):
    CD_bool += [False]

AD_bool = []
for i in range(n_start_ad-1):
    AD_bool += [False]

for i in range(n_start_ad-1, n_steps):
    AD_bool += [True]



shots_var = 100000
runs = 100
sampling_size = 6

string_list = []
for i in range(runs):
    result = cudaq.sample(qc, n, dt, list_2, list_4, thetas, lamb_list, CD_bool, AD_bool, shots_count=shots_var)
    top_n_bitstrings = [bs for bs, count in sorted(result.items(), key=lambda x: x[1], reverse=True)[:sampling_size]]
    string_list += [top_n_bitstrings]

print(string_list)

[['0000101', '1010000', '1111010', '0101111', '1110111', '1011101'], ['0000101', '1111010', '1010000', '0101111', '1011101', '1110111'], ['0000101', '1111010', '0101111', '1010000', '1110111', '1011101'], ['0101111', '1111010', '0000101', '1010000', '0100010', '0001000'], ['0101111', '1010000', '1111010', '0000101', '1110111', '0001000'], ['0101111', '1111010', '0000101', '1010000', '1110111', '0100010'], ['1010000', '0000101', '0101111', '1111010', '0100010', '1110111'], ['0101111', '1010000', '1111010', '0000101', '1110111', '1011101'], ['0101111', '1010000', '0000101', '1111010', '1011101', '0100010'], ['0101111', '1010000', '0000101', '1111010', '0001000', '0100010'], ['1111010', '0000101', '1010000', '0101111', '1110111', '0001000'], ['1010000', '1111010', '0101111', '0000101', '0001000', '0100010'], ['1111010', '1010000', '0101111', '0000101', '1110111', '0001000'], ['0101111', '1111010', '0000101', '1010000', '1011101', '0100010'], ['0101111', '1010000', '0000101', '1111010', '1

In [43]:
def labs_energy(bitstring, n):
    """
    Calculate LABS energy for a bitstring.
    bitstring: string like "01101" or list/array of 0s and 1s
    n: number of qubits
    Returns: energy value
    """
    # Convert bitstring to ±1 sequence
    if isinstance(bitstring, str):
        s = [1 if b == '0' else -1 for b in bitstring]
    else:
        s = [1 if b == 0 else -1 for b in bitstring]
    
    # Calculate energy E = sum of C_k^2
    energy = 0
    for k in range(1, n):
        # Calculate C_k = sum of s_i * s_{i+k}
        ck = sum(s[i] * s[i+k] for i in range(n-k))
        energy += ck**2
    
    return energy


e = []
for j in range(len(string_list)):
    for i in string_list[j]:
        e += [labs_energy(i, 7)]



print(e)
print(e.count(19))

[11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11, 19, 19, 11, 11, 11, 11,

In [44]:
print(string_list)

[['0000101', '1010000', '1111010', '0101111', '1110111', '1011101'], ['0000101', '1111010', '1010000', '0101111', '1011101', '1110111'], ['0000101', '1111010', '0101111', '1010000', '1110111', '1011101'], ['0101111', '1111010', '0000101', '1010000', '0100010', '0001000'], ['0101111', '1010000', '1111010', '0000101', '1110111', '0001000'], ['0101111', '1111010', '0000101', '1010000', '1110111', '0100010'], ['1010000', '0000101', '0101111', '1111010', '0100010', '1110111'], ['0101111', '1010000', '1111010', '0000101', '1110111', '1011101'], ['0101111', '1010000', '0000101', '1111010', '1011101', '0100010'], ['0101111', '1010000', '0000101', '1111010', '0001000', '0100010'], ['1111010', '0000101', '1010000', '0101111', '1110111', '0001000'], ['1010000', '1111010', '0101111', '0000101', '0001000', '0100010'], ['1111010', '1010000', '0101111', '0000101', '1110111', '0001000'], ['0101111', '1111010', '0000101', '1010000', '1011101', '0100010'], ['0101111', '1010000', '0000101', '1111010', '1